# ToysAD-8K Official Prediction Summary

**Notebook file:** `10a-finalize-toysad8k-official-prediction-summary.ipynb`

This CPU-only notebook joins the completed seen-test and unseen-test prediction locks into one minimal summary. It opens no RGB or evaluator data and computes no accuracy metric. Its only purpose is to prove that both role-specific model runs used the same frozen method, review policy, and completed internal-lock protocol.

## Finalization contract

- The private datasets `vpi-toysad-official-seen-prediction` and `vpi-toysad-official-unseen-prediction` must both be mounted.
- Exactly one verified prediction lock must exist for each official cohort.
- Method, policy, cohort membership, and internal-lock receipt hashes must match.
- The output contains aggregate lock metadata only.
- A bounded ClearML finalizer runs after the summary and its artifact manifest are complete.
- Evaluator datasets, labels, source identifiers, RGB files, masks, and embeddings are not mounted or uploaded.

In [ ]:
from __future__ import annotations

from hashlib import sha256
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image as NotebookImage, display

sys.dont_write_bytecode = True
NOTEBOOK_FILE = "10a-finalize-toysad8k-official-prediction-summary.ipynb"
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"
OUTPUT_ROOT = Path(os.getenv(
    "VPI_OUTPUT_ROOT",
    "/kaggle/working/toysad8k_official_prediction_summary" if EXECUTION_ENVIRONMENT == "kaggle" else ".local/runs/toysad8k_official_prediction_summary",
)).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:

def recovery_archives(root: Path, filename: str) -> list[Path]:
    """Find a ZIP or byte-preserved Kaggle upload; the caller verifies its hashes."""
    return sorted({path.resolve() for name in (filename, filename + ".bin")
                   for path in root.rglob(name) if path.is_file()}, key=lambda path: path.as_posix())

def kaggle_roots() -> list[Path]:
    """Return dataset and notebook mounts without their container directories."""
    root = Path("/kaggle/input")
    if not root.is_dir():
        return []
    candidates = [path for path in root.iterdir()
                  if path.is_dir() and path.name not in {"datasets", "notebooks", "models"}]
    for pattern in ("datasets/*/*", "notebooks/*/*", "models/*/*/*/*"):
        for path in root.glob(pattern):
            if not path.is_dir():
                continue
            versions = path / "versions"
            if versions.is_dir():
                candidates.extend(child for child in versions.iterdir() if child.is_dir())
            else:
                candidates.append(path)
    return sorted({path.resolve() for path in candidates}, key=lambda path: path.as_posix())


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *kaggle_roots()))
    for candidate in candidates:
        if (candidate / "scripts/run_official_inference.py").is_file():
            return candidate.resolve()
        if (candidate / "code/scripts/run_official_inference.py").is_file():
            return (candidate / "code").resolve()
    raise FileNotFoundError("Could not locate the Visual Peer Inspector code snapshot")


def find_named_file(name: str, environment_name: str) -> Path:
    configured = os.getenv(environment_name)
    if configured and Path(configured).expanduser().is_file():
        return Path(configured).expanduser().resolve()
    matches = sorted({path.resolve() for root in kaggle_roots() for path in root.rglob(name) if path.is_file()})
    if not matches:
        raise FileNotFoundError(f"Could not locate {name}")
    digests = {sha256(path.read_bytes()).hexdigest() for path in matches}
    if len(digests) != 1:
        raise ValueError(f"Mounted copies of {name} differ")
    return matches[0]


def find_release_recovery_pair(role: str) -> tuple[Path, Path] | None:
    configured = os.getenv("VPI_RESUME_ROOT")
    sources = [Path(configured).expanduser()] if configured else kaggle_roots()
    bundles = sorted({
        path.resolve()
        for source in sources
        for path in recovery_archives(source, f"official-{role}-release-recovery.zip")
        if path.is_file()
    })
    manifests = sorted({
        path.resolve()
        for source in sources
        for path in source.rglob(f"official-{role}-release-recovery-manifest.json")
        if path.is_file()
    })
    if not bundles and not manifests:
        return None
    if len(bundles) != 1 or len(manifests) != 1:
        raise ValueError("VPI_RESUME_ROOT must contain exactly one official release recovery pair")
    return bundles[0], manifests[0]


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source_root = str(CODE_ROOT / "src")
    existing = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source_root if not existing else source_root + os.pathsep + existing
    return environment


def read_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)


def enable_tracking(environment: dict[str, str]) -> str:
    access = read_secret("CLEARML_API_ACCESS_KEY")
    secret = read_secret("CLEARML_API_SECRET_KEY")
    if not access or not secret:
        return "disabled"
    environment.update({
        "CLEARML_API_ACCESS_KEY": access,
        "CLEARML_API_SECRET_KEY": secret,
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
    })
    return "online"


def prepare_tracking_runtime(mode: str, environment: dict[str, str]) -> str:
    if mode != "online":
        return mode
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-tracking-lock.txt")],
            check=True,
            timeout=180,
            env=environment,
        )
    except Exception as error:
        print(f"ClearML dependency setup was unavailable: {type(error).__name__}")
        return "disabled"
    return mode

In [ ]:
def find_prediction_lock(cohort_id: str) -> Path:
    matches = []
    for root in kaggle_roots():
        for path in root.rglob("prediction_lock.json"):
            try:
                record = json.loads(path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if record.get("cohort_id") == cohort_id:
                matches.append(path.resolve())
    if len(matches) != 1:
        raise ValueError(f"Expected one completed prediction lock for {cohort_id}")
    return matches[0]

## Verify and combine the two prediction locks

In [ ]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/official-evaluation/10a-finalize-toysad8k-official-prediction-summary.ipynb",
)
METHOD_FREEZE = find_named_file("method_freeze.json", "VPI_METHOD_FREEZE")
APPLICATION_CONTRACT = find_named_file("application_output_contract.json", "VPI_APPLICATION_CONTRACT")
POLICY_FREEZE = find_named_file("review_policy_freeze.json", "VPI_POLICY_FREEZE")
LOCK_RESULT_RECEIPT = find_named_file("internal_lock_result_receipt.json", "VPI_LOCK_RESULT_RECEIPT")
SEEN_LOCK = find_prediction_lock("toysad8k-official-seen-test-715-v1")
UNSEEN_LOCK = find_prediction_lock("toysad8k-official-unseen-test-2437-v1")

environment = code_environment()
tracking_mode = prepare_tracking_runtime(enable_tracking(environment), environment)
command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_official_inference.py"),
    "--cohort", "seen",
    "--run-mode", "finalize",
    "--method-freeze", str(METHOD_FREEZE),
    "--application-contract", str(APPLICATION_CONTRACT),
    "--policy-freeze", str(POLICY_FREEZE),
    "--lock-result-receipt", str(LOCK_RESULT_RECEIPT),
    "--seen-prediction-lock", str(SEEN_LOCK),
    "--unseen-prediction-lock", str(UNSEEN_LOCK),
    "--output-root", str(OUTPUT_ROOT),
    "--device", "cpu",
    "--clearml-mode", tracking_mode,
]
for mounted in kaggle_roots():
    command.extend(["--mounted-root", str(mounted)])
subprocess.run(command, check=True, timeout=900, env=environment)

## Inspect the aggregate-only result

In [ ]:
SAFE_ROOT = OUTPUT_ROOT / "official-safe"
summary = json.loads((SAFE_ROOT / "official_inference_summary.json").read_text(encoding="utf-8"))
table = pd.read_csv(SAFE_ROOT / "official_prediction_lock_summary.csv")
display(summary)
display(table)
display(NotebookImage(filename=str(SAFE_ROOT / "official_prediction_lock_summary.png")))

from visual_peer_inspector.frozen_resource import verify_frozen_resource

publish_root = OUTPUT_ROOT / "publish" / "official-prediction-summary"
resource_record = verify_frozen_resource(
    publish_root,
    expected_stage="official_prediction_summary_release",
    expected_files={"official_inference_summary.json"},
)
display(resource_record)

## Publication boundary

Publish only `publish/official-prediction-summary` as the private dataset `vpi-toysad-official-prediction-summary`. Notebook 10B mounts this minimal resource together with the two role-specific prediction datasets and the two physically separate evaluator datasets. Do not publish the complete finalizer output as a model input.